<a href="https://colab.research.google.com/github/fabricioagn/trab_IA/blob/main/trab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install google-genai panel python-dotenv
!pip install jupyter_bokeh

In [36]:
import os
import panel as pn
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Configura a extensão do Panel (funciona no Colab e Localmente)
pn.extension('katex', template='bootstrap')

# 1. Carrega as variáveis do arquivo .env (se existir localmente)
load_dotenv()

# 2. Tenta pegar a chave das variáveis de ambiente (.env)
GOOGLE_API_KEY = os.getenv('chave1')

# 3. Se não encontrar no .env, tenta pegar dos Secrets do Colab
if not GOOGLE_API_KEY:
    try:
        from google.colab import userdata
        GOOGLE_API_KEY = userdata.get('chave1')
    except ImportError:
        pass

# Validação da Chave
if not GOOGLE_API_KEY:
    raise ValueError("Chave 'Gemini_API_Key' não foi encontrada nem no .env e nem nos Secrets do Colab.")

# Inicializa o cliente com a chave obtida
client = genai.Client(api_key=GOOGLE_API_KEY)

# Função de chamada da API com o modelo corrigido para 'gemini-2.5-flash'
def get_completion_from_messages(messages, model="gemini-3.6-flash", temperature=0):
    genai_messages = []
    system_instruction = None

    for msg in messages:
        role = msg['role']
        content = msg['content']

        if role == 'system':
            system_instruction = content
        elif role == 'assistant':
            genai_messages.append(types.Content(role='model', parts=[types.Part.from_text(text=content)]))
        elif role == 'user':
            genai_messages.append(types.Content(role='user', parts=[types.Part.from_text(text=content)]))

    config = types.GenerateContentConfig(
        temperature=temperature,
        system_instruction=system_instruction
    )

    response = client.models.generate_content(
        model=model,
        contents=genai_messages,
        config=config,
    )
    return response.text

context = [
    {
        'role': 'system',
        'content': """
Você é o TechBot, assistente virtual oficial da TechFix Pro.

--- PERSONALIDADE ---
Você é profissional, educado, objetivo e prestativo. Responda sempre em Português do Brasil.

--- CONHECIMENTO INTERNO (DADOS PRIVADOS DO SERVIÇO) ---
- Empresa: TechFix Pro - Suporte e Garantia Corporativa.
- Planos de Suporte:
  * Plano Básico: Atendimento remoto em até 24 horas úteis. R$ 150/mês por equipamento.
  * Plano Avançado: Atendimento presencial em até 4 horas úteis. R$ 350/mês por equipamento.
- Cobertura de Garantia:
  * Cobre falhas de hardware, substituição de telas e baterias originais.
  * NÃO cobre danos por derramamento de líquidos ou quedas acidentais.
- Procedimento de Devolução/Troca:
  * O cliente deve solicitar via portal com a Nota Fiscal. O prazo máximo para abertura de chamado de troca é de 7 dias corridos após o recebimento.
- Visita Técnica Avulsa:
  * Custa R$ 200,00 por hora técnica para clientes fora do Plano Avançado.

--- REGRAS RÍGIDAS DE RESPOSTA (NÃO ALUCINAÇÃO) ---
1. Responda ÚNICA e EXCLUSIVAMENTE com base no Conhecimento Interno fornecido acima.
2. Se o usuário perguntar algo que NÃO está nas informações acima (ex: aplicativo móvel, endereço físico, desconto, marcas específicas), responda exatamente:
   "Esta informação não consta na minha base de conhecimento interna."
3. NUNCA tente inventar ou supor respostas fora desse contexto.
"""
    }
]

# Variáveis globais de estado
messages = context.copy()
user_question_count = 0

# Componentes da Interface Panel
inp = pn.widgets.TextInput(placeholder='Digite sua pergunta aqui e clique em Enviar...', width=500)
button_conversation = pn.widgets.Button(name="Enviar", button_type="primary", width=100)
conversation_panel = pn.Column(width=650)

def collect_messages(event):
    global messages, user_question_count

    prompt = inp.value
    if not prompt.strip():
        return

    inp.value = ''
    user_question_count += 1

    user_prompt = prompt

    # Injeta instrução de encerramento e resumo na 3ª pergunta
    if user_question_count == 3:
        user_prompt = (
            f"{prompt}\n\n"
            "[INSTRUÇÃO DO SISTEMA PARA ESTA RESPOSTA]: Esta é a 3ª e última pergunta do usuário. "
            "Responda à pergunta normalmente. Em seguida, inclua o cabeçalho '--- RESUMO DO ATENDIMENTO ---', "
            "apresente um resumo breve das 3 perguntas e respostas fornecidas ao longo do chat e encerre informando de forma cortês que o atendimento foi encerrado."
        )

    messages.append({'role': 'user', 'content': user_prompt})

    try:
        response = get_completion_from_messages(messages)
        messages.append({'role': 'assistant', 'content': response})

        user_row = pn.Row('**Você:**', pn.pane.Markdown(prompt, width=550))
        bot_row = pn.Row(
            '**TechBot:**',
            pn.pane.Markdown(response, width=550),
            styles={'background-color': '#F0F4F8', 'padding': '10px', 'border-radius': '5px', 'margin-bottom': '10px'}
        )

        conversation_panel.append(user_row)
        conversation_panel.append(bot_row)

        if user_question_count >= 3:
            inp.disabled = True
            button_conversation.disabled = True
            conversation_panel.append(pn.pane.Markdown('---\n🔒 **Atendimento finalizado. Limite de 3 perguntas atingido.**'))

    except Exception as e:
        user_question_count -= 1
        messages.pop()
        conversation_panel.append(pn.Row('**ERRO:**', pn.pane.Markdown(f"```{str(e)}```", width=550)))

button_conversation.on_click(collect_messages)

app = pn.Column(
    pn.pane.Markdown("## 🤖 TechBot - Assistência TechFix Pro"),
    pn.Row(inp, button_conversation),
    conversation_panel
)

app.servable()

Column
    [0] Markdown(str)
    [1] Row
        [0] TextInput(placeholder='Digite sua pergunta a..., width=500)
        [1] Button(button_type='primary', color='primary', label='Enviar', name='Enviar', width=100)
    [2] Column(width=650)